In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/OE/Kerendia_OE_2.0/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/OE/Kerendia_OE_2.0/00_Helper_Notebooks/00_config"

In [0]:
# df_nba_all = spark.read.format("delta").load(
#     "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/oe_journey/oe_volume_space/notebook_output/oe_final_nba_all_features_v1"
# )

df_nba_all = spark.read.format("delta").table("`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_oe_journey.oe_final_nba_all_features_v2")

In [0]:
from pyspark.sql import functions as F

pct_cols = [c for c in df_nba_all.columns if "PCT" in c]

# Single pass: get max of every PCT column at once
max_row = df_nba_all.agg(*[F.max(F.col(c)).alias(c) for c in pct_cols]).first()

# Keep only columns whose max exceeds 1
filtered_cols = [c for c in pct_cols if max_row[c] is not None and max_row[c] > 1]

print(f"PCT columns with at least one value > 1: {len(filtered_cols)}")

df_nba_all.select(*filtered_cols).display()